In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input


# =========================
# 2. TEXT DATA
# =========================
sport_text = """
Sports have always been an important part of human life. From childhood to old age, people connect with sports in different ways. Sports teach discipline, teamwork, and confidence. They build both physical strength and mental toughness.

Athletes wake up early every morning and train hard to improve their skills. Running, stretching, and practice drills are part of their daily routine. Consistency and dedication separate great athletes from average players. Success in sports requires patience and focus.

Team sports like football and cricket teach cooperation and trust. Every player has a role in the team. Some players attack, some defend, and some guide the team. Winning is possible only when players work together.

Individual sports test self-control and mental strength. Athletes compete not only against others but also against themselves. Focus, balance, and confidence play a major role in performance. Mental preparation is as important as physical fitness.

Matches are full of excitement and pressure. One small mistake can change the result of the game. Fans cheer loudly and support their teams with passion. Moments of victory create unforgettable memories.

Sports also teach how to handle failure. Losing a match can be painful, but it helps athletes learn from mistakes. Every failure becomes a lesson for future success. Strong athletes return better after defeat.

Coaches guide athletes and motivate them during difficult times. A good coach understands strategy, technique, and psychology. Trust between coach and player improves performance.

Fitness is one of the biggest benefits of sports. Regular training improves stamina, strength, and flexibility. A healthy body supports a sharp and active mind.

Sports bring people together across cultures and countries. International tournaments unite nations and create emotional connections. Sports inspire young people to dream big and work hard.

In the end, sports are more than just games. They teach life lessons such as discipline, teamwork, patience, and courage. Sports shape character and build confidence for life.
"""


# =========================
# 3. TOKENIZATION
# =========================
tokenizer = Tokenizer()
tokenizer.fit_on_texts([sport_text])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size:", total_words)


# =========================
# 4. CREATE N-GRAM SEQUENCES
# =========================
input_sequences = []

for line in sport_text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])


# =========================
# 5. PADDING
# =========================
max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_seq_len,
    padding="pre"
)


# =========================
# 6. SPLIT X AND y
# =========================
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)


# =========================
# 7. BUILD MODEL (FIXED)
# =========================
model = Sequential([
    Input(shape=(X.shape[1],)),   
    Embedding(total_words, 100),
    LSTM(150),
    Dense(total_words, activation="softmax")
])

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


# =========================
# 8. MODEL SUMMARY
# =========================
model.summary()


# =========================
# 9. TRAIN MODEL
# =========================
model.fit(X, y, epochs=50, verbose=1)


# =========================
# 10. TEXT GENERATION FUNCTION
# =========================
def generate_text(seed_text, next_words, model, tokenizer, max_seq_len):

    index_to_word = {index: word for word, index in tokenizer.word_index.items()}

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=max_seq_len - 1,
            padding="pre"
        )

        predictions = model.predict(token_list, verbose=0)[0]
        predicted_index = np.argmax(predictions)

        seed_text += " " + index_to_word[predicted_index]

    return seed_text


# =========================
# 11. GENERATE TEXT
# =========================
print("\nGenerated Text:\n")

print(generate_text(
    seed_text="sports have",
    next_words=20,
    model=model,
    tokenizer=tokenizer,
    max_seq_len=max_seq_len
))

Vocabulary Size: 194
X shape: (307, 39)
y shape: (307, 194)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 39, 100)        │        19,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 194)            │        29,294 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 199,294 (778.49 KB)

 Trainable params: 199,294 (778.49 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.0423 - loss: 5.2659
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.0749 - loss: 5.1815
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.0749 - loss: 5.0423
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.0749 - loss: 4.9905
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.0749 - loss: 4.9420
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.0749 - loss: 4.8986
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.0814 - loss: 4.8343
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.0782 - loss: 4.7488
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.0814 - loss: 4.6690
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.0814 - loss: 4.5518
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.0782 - loss: 4.4564
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: